# L63 reconstruction examples (`obs_interval = 10`)

Example reconstructions of the Lorenz-63 test windows, comparing the neural schemes (**D-Unet**, **CFM**, **SDA2**, **D-Unet+SDA2**) with the ETKS (30 members), which is used as the reference.

**How the windows are chosen.** Every test window is ranked by the RMSE of the **ETKS ensemble mean** against the truth (per window: time-RMSE of each variable, averaged over X, Y, Z). For each case the notebook then shows

* **good**: the 3 windows with the lowest ETKS RMSE,
* **medium**: the 3 windows around the median ETKS RMSE,
* **bad**: the 3 windows with the highest ETKS RMSE.

**How to read the figures.** One figure per window, one column per scheme, one row per variable (X, Y, Z). Black: truth. Coloured line: the scheme's estimate (ensemble mean for ensemble schemes, single pass for D-Unet). Shaded band: ensemble mean ± 1 standard deviation across the 30 members (ETKS, CFM, SDA2, D-Unet+SDA2; D-Unet has no spread). Dashed grey: the ETKS mean, repeated in the neural columns for comparison. Dots: the noisy observations the schemes received (all three variables, every 10 steps).

The reconstructions are computed once by `l63_reconstructions.load_or_compute` and cached in `experiments/l63_da_table2/reconstructions_obs10_<case>.npz`. The ETKS ensemble is re-drawn for this notebook, so its window RMSE matches the Table 2 value only up to realisation noise. Neural hyper-parameters (SDA2 guidance weight, hybrid start time and weight) are the ones tuned for Table 2.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import l63_reconstructions as rec

OBS_INTERVAL = 10
DT = 0.01
CASES = {"s0": "S0 (perfect model)", "s1": "S1 (imperfect model)"}
METHODS = rec.METHODS
COLORS = {"ETKS": "#2a78d6", "D-Unet": "#eb6834", "CFM": "#1baf7a", "SDA2": "#6250d6", "D-Unet+SDA2": "#e87ba4"}
INK, INK_SOFT, GRID, SURFACE, REFERENCE, OBS_COLOR = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#4a4945", "#0b0b0b"
GROUP_COLORS = {"good": "#1baf7a", "medium": "#eda100", "bad": "#e34948"}
VARIABLES = ["X", "Y", "Z"]
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT, "xtick.color": INK_SOFT,
                     "ytick.color": INK_SOFT, "text.color": INK, "font.size": 9})

## Load the reconstructions and rank the windows by the ETKS error

In [ ]:
data = {case: rec.load_or_compute(case, OBS_INTERVAL) for case in CASES}

window_rmse = {}
for case, d in data.items():
    truth = d["truth"]
    window_rmse[case] = {m: rec.per_window_rmse(d[f"members_{m}"].mean(axis=1), truth) for m in METHODS}


def select_windows(etks_rmse, n=3):
    order = np.argsort(etks_rmse)
    mid = len(order) // 2
    return {"good": order[:n].tolist(),
            "medium": order[mid - n // 2: mid - n // 2 + n].tolist(),
            "bad": order[-n:][::-1].tolist()}


selection = {case: select_windows(window_rmse[case]["ETKS"]) for case in CASES}
summary = pd.DataFrame({case: {m: window_rmse[case][m].mean() for m in METHODS} for case in CASES}).round(3)
summary.columns = [f"mean window RMSE, {CASES[c]}" for c in CASES]
summary

## Where the selected windows sit

Each dot is one of the 200 test windows: ETKS window RMSE on the x-axis, a neural scheme's window RMSE on the y-axis. Points below the diagonal are windows where the neural scheme beats the ETKS. Coloured rings mark the 9 windows shown below (green = good, amber = medium, red = bad).

In [ ]:
def plot_overview(case):
    fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharex=True, sharey=True)
    x = window_rmse[case]["ETKS"]
    lim = 1.05 * max(x.max(), max(window_rmse[case][m].max() for m in METHODS[1:]))
    for ax, m in zip(axes, METHODS[1:]):
        y = window_rmse[case][m]
        ax.scatter(x, y, s=12, color=COLORS[m], alpha=0.55, linewidths=0)
        for group, idx in selection[case].items():
            ax.scatter(x[idx], y[idx], s=70, facecolors="none", edgecolors=GROUP_COLORS[group], linewidths=1.8)
        ax.plot([0, lim], [0, lim], color=REFERENCE, lw=1, ls="--")
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_title(m, loc="left", color=INK)
        ax.set_xlabel("ETKS window RMSE")
        ax.grid(color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].set_ylabel("neural window RMSE")
    fig.suptitle(f"{CASES[case]}: per-window RMSE against the ETKS", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


for case in CASES:
    plot_overview(case)

## Plotting one window

In [ ]:
def style_axis(ax):
    ax.grid(color=GRID, lw=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def plot_window(case, idx, group):
    d = data[case]
    truth, obs = d["truth"][idx], d["obs"][idx]
    t = np.arange(truth.shape[0]) * DT
    observed = ~np.isnan(obs[:, 0])
    etks_mean = d["members_ETKS"][idx].mean(axis=0)
    fig, axes = plt.subplots(3, len(METHODS), figsize=(17, 7.2), sharex=True, sharey="row")
    for col, m in enumerate(METHODS):
        members = d[f"members_{m}"][idx]
        mean = members.mean(axis=0)
        std = members.std(axis=0, ddof=1) if members.shape[0] > 1 else None
        for row in range(3):
            ax = axes[row, col]
            if std is not None:
                ax.fill_between(t, mean[:, row] - std[:, row], mean[:, row] + std[:, row],
                                color=COLORS[m], alpha=0.25, linewidth=0)
            if m != "ETKS":
                ax.plot(t, etks_mean[:, row], color=REFERENCE, lw=1.9, ls="--", zorder=4)
            ax.plot(t, truth[:, row], color=INK, lw=1.4, zorder=3)
            ax.plot(t, mean[:, row], color=COLORS[m], lw=1.0, zorder=5)
            ax.scatter(t[observed], obs[observed, row], s=30, color=OBS_COLOR, edgecolors=SURFACE,
                       linewidths=0.6, zorder=6)
            style_axis(ax)
            if col == 0:
                ax.set_ylabel(VARIABLES[row])
            if row == 2:
                ax.set_xlabel("time")
        axes[0, col].set_title(f"{m}   RMSE {window_rmse[case][m][idx]:.2f}", loc="left", color=INK)
    handles = [Line2D([], [], color=INK, lw=1.4, label="truth"),
               Line2D([], [], color=INK_SOFT, lw=1.0, label="estimate (ensemble mean)"),
               Patch(facecolor=INK_SOFT, alpha=0.25, label="ensemble mean ± 1 std"),
               Line2D([], [], color=REFERENCE, lw=1.9, ls="--", label="ETKS mean (reference)"),
               Line2D([], [], color=OBS_COLOR, marker="o", ls="", markersize=6, label="observations")]
    fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{CASES[case]} | {group} | test window {idx} | ETKS RMSE {window_rmse[case]['ETKS'][idx]:.2f}",
                 x=0.01, ha="left", color=GROUP_COLORS[group])
    fig.tight_layout(rect=(0, 0.04, 1, 0.95))
    plt.show()


def plot_overlay(case, idx, group):
    d = data[case]
    truth, obs = d["truth"][idx], d["obs"][idx]
    t = np.arange(truth.shape[0]) * DT
    observed = ~np.isnan(obs[:, 0])
    fig, axes = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True)
    for row, ax in enumerate(axes):
        for m in METHODS:
            members = d[f"members_{m}"][idx]
            mean = members.mean(axis=0)
            if members.shape[0] > 1:
                std = members.std(axis=0, ddof=1)
                ax.fill_between(t, mean[:, row] - std[:, row], mean[:, row] + std[:, row],
                                color=REFERENCE if m == "ETKS" else COLORS[m], alpha=0.12, linewidth=0)
            if m == "ETKS":
                ax.plot(t, mean[:, row], color=REFERENCE, lw=1.9, ls="--", zorder=4)
            else:
                ax.plot(t, mean[:, row], color=COLORS[m], lw=1.0, zorder=5)
        ax.plot(t, truth[:, row], color=INK, lw=1.6, zorder=3)
        ax.scatter(t[observed], obs[observed, row], s=34, color=OBS_COLOR, edgecolors=SURFACE,
                   linewidths=0.6, zorder=6)
        style_axis(ax)
        ax.set_ylabel(VARIABLES[row])
    axes[-1].set_xlabel("time")
    handles = [Line2D([], [], color=INK, lw=1.6, label="truth"),
               Line2D([], [], color=OBS_COLOR, marker="o", ls="", markersize=6, label="observations"),
               Line2D([], [], color=REFERENCE, lw=1.9, ls="--",
                      label=f"ETKS (RMSE {window_rmse[case]['ETKS'][idx]:.2f})")]
    handles += [Line2D([], [], color=COLORS[m], lw=1.0, label=f"{m} (RMSE {window_rmse[case][m][idx]:.2f})")
                for m in METHODS[1:]]
    fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{CASES[case]} | {group} | test window {idx} | all schemes, shaded = ensemble mean ± 1 std",
                 x=0.01, ha="left", color=GROUP_COLORS[group])
    fig.tight_layout(rect=(0, 0.07, 1, 0.96))
    plt.show()

## S0 (perfect model)

In [ ]:
case = "s0"
pd.DataFrame({group: [f"window {i} (ETKS {window_rmse[case]['ETKS'][i]:.2f})" for i in idx]
              for group, idx in selection[case].items()}, index=[1, 2, 3])

### S0 (perfect model): good reconstructions (by ETKS error)

In [ ]:
for idx in selection["s0"]["good"]:
    plot_window("s0", idx, "good")
plot_overlay("s0", selection["s0"]["good"][0], "good")

### S0 (perfect model): medium reconstructions (by ETKS error)

In [ ]:
for idx in selection["s0"]["medium"]:
    plot_window("s0", idx, "medium")
plot_overlay("s0", selection["s0"]["medium"][0], "medium")

### S0 (perfect model): bad reconstructions (by ETKS error)

In [ ]:
for idx in selection["s0"]["bad"]:
    plot_window("s0", idx, "bad")
plot_overlay("s0", selection["s0"]["bad"][0], "bad")

## S1 (imperfect model)

In [ ]:
case = "s1"
pd.DataFrame({group: [f"window {i} (ETKS {window_rmse[case]['ETKS'][i]:.2f})" for i in idx]
              for group, idx in selection[case].items()}, index=[1, 2, 3])

### S1 (imperfect model): good reconstructions (by ETKS error)

In [ ]:
for idx in selection["s1"]["good"]:
    plot_window("s1", idx, "good")
plot_overlay("s1", selection["s1"]["good"][0], "good")

### S1 (imperfect model): medium reconstructions (by ETKS error)

In [ ]:
for idx in selection["s1"]["medium"]:
    plot_window("s1", idx, "medium")
plot_overlay("s1", selection["s1"]["medium"][0], "medium")

### S1 (imperfect model): bad reconstructions (by ETKS error)

In [ ]:
for idx in selection["s1"]["bad"]:
    plot_window("s1", idx, "bad")
plot_overlay("s1", selection["s1"]["bad"][0], "bad")

## RMSE of the nine selected windows

In [ ]:
rows = []
for case in CASES:
    for group, idx in selection[case].items():
        for i in idx:
            rows.append({"case": case.upper(), "group": group, "window": i,
                         **{m: round(float(window_rmse[case][m][i]), 3) for m in METHODS}})
pd.DataFrame(rows)